# CLTV С OOT ВЫБОРКОЙ - БЕЗ CHURN, БЫСТРОЕ ОБУЧЕНИЕ
# Два сегмента: small и large_and_middle
# Train / Validation / OOT (Out-of-Time Test)

In [ ]:
# %% ИМПОРТЫ И НАСТРОЙКИ
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from dateutil.relativedelta import relativedelta
import os
import warnings
from collections import defaultdict, deque
import logging
import pickle
import json
from pathlib import Path

from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option('display.max_columns', None)
warnings.filterwarnings('ignore')
plt.style.use('default')
sns.set_palette("husl")

print("Импорты загружены")

In [ ]:
# %% КОНФИГУРАЦИЯ С OOT
class Config:
    # Пути к Parquet файлам
    DATA_DIR = Path("data")
    TRAIN_PATH = DATA_DIR / "train_data.parquet"
    PROD_PATH = DATA_DIR / "prod_data.parquet"
    
    MODEL_DIR = Path("models_oot")
    MODEL_VERSION = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # ============== ВАЖНО: РАЗДЕЛЕНИЕ НА 3 ВЫБОРКИ ==============
    TRAIN_CUTOFF = "2025-01-31"       # Обучение: до 31 января 2025
    VALIDATION_CUTOFF = "2025-03-31"  # Валидация: февраль-март 2025
    # OOT (Test): после 31 марта 2025
    # ===========================================================
    
    FORECAST_START = "2025-10-31"
    HORIZON_MONTHS = 6
    MIN_SAMPLES_PER_SEGMENT = 1000
    
    CATEGORICAL_FEATURES = ['QUALITY_CODE', 'SUBJECT_KIND_ID', 'EC_SECTOR_ID']
    BASE_FEATURES = [
        'MARGIN', 'MARGIN_LAG1', 'MARGIN_LAG2', 'MARGIN_LAG3',
        'MARGIN_AVG_1M_LAG', 'MARGIN_AVG_2M_LAG', 'MARGIN_AVG_3M_LAG',
        'MARGIN_AVG_6M_LAG', 'MARGIN_AVG_12M_LAG', 'MARGIN_STDDEV_12M_LAG',
        'MARGIN_GROWTH_RATE_3M', 'MONTH_OF_YEAR', 'QUARTER_OF_YEAR', 'TENURE_MONTHS'
    ]
    
    # Маппинг сегментов
    SEGMENT_MAPPING = {
        '1026': 'small',              # MICRO
        '1027': 'small',              # SMALL
        '1022': 'large_and_middle',   # MIDDLE
        '1023': 'large_and_middle',   # LARGE
        '1040': 'large_and_middle',   # → large_and_middle
        '1028': 'large_and_middle',
    }
    
    # Быстрые параметры CatBoost
    CATBOOST_PARAMS = {
        'iterations': 500,
        'depth': 4,
        'learning_rate': 0.05,
        'l2_leaf_reg': 3,
        'random_seed': 42,
        'loss_function': 'RMSE',
        'verbose': 100,
        'early_stopping_rounds': 50
    }
    
    @classmethod
    def ensure_directories(cls):
        cls.MODEL_DIR.mkdir(parents=True, exist_ok=True)
        (cls.MODEL_DIR / cls.MODEL_VERSION).mkdir(parents=True, exist_ok=True)

Config.ensure_directories()
print(f"Версия: {Config.MODEL_VERSION}")
print(f"\n{'='*70}")
print("РАЗДЕЛЕНИЕ ДАННЫХ:")
print(f"{'='*70}")
print(f"  Train:      до {Config.TRAIN_CUTOFF}")
print(f"  Validation: {Config.TRAIN_CUTOFF} - {Config.VALIDATION_CUTOFF}")
print(f"  OOT (Test): после {Config.VALIDATION_CUTOFF}")
print(f"{'='*70}")

print(f"\nМаппинг сегментов:")
for old_seg, new_seg in sorted(Config.SEGMENT_MAPPING.items()):
    print(f"  {old_seg} → {new_seg}")

In [ ]:
# %% ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
def read_parquet(path):
    """Чтение Parquet файлов"""
    if not os.path.exists(path):
        logger.warning(f"Файл не найден: {path}")
        return pd.DataFrame()
    return pd.read_parquet(path)

def fix_categorical_features(df, cat_features):
    df_fixed = df.copy()
    for col in cat_features:
        if col in df_fixed.columns:
            df_fixed[col] = df_fixed[col].fillna('UNKNOWN').astype(str)
            df_fixed[col] = df_fixed[col].str.replace('.0', '', regex=False)
    return df_fixed

def stabilize_target(y):
    """Log-трансформация таргета"""
    return np.sign(y) * np.log1p(np.abs(y))

def inverse_stabilize_target(y_stable):
    """Обратная трансформация"""
    return np.sign(y_stable) * (np.exp(np.abs(y_stable)) - 1)

print("Функции загружены")

In [ ]:
# %% ЗАГРУЗКА ДАННЫХ ИЗ PARQUET
logger.info("Загрузка данных из Parquet...")

train = read_parquet(Config.TRAIN_PATH)
prod = read_parquet(Config.PROD_PATH)

print(f"\nОбучающая: {len(train):,} записей, {train['CLIENT_ID'].nunique():,} клиентов")
print(f"Продакшн: {len(prod):,} записей")

if train.empty or prod.empty:
    print("\n⚠️  ВНИМАНИЕ: Файлы не загружены!")
    print("Запустите сначала notebook 'data_loader.ipynb'")
else:
    print("\n✅ Данные успешно загружены")

In [ ]:
# %% ОБЪЕДИНЕНИЕ СЕГМЕНТОВ
logger.info("Объединение сегментов...")

print("\n" + "="*70)
print("ОРИГИНАЛЬНЫЕ СЕГМЕНТЫ")
print("="*70)
print(train['SEGMENT_ID'].value_counts().sort_index())

# Преобразование к строке и применение маппинга
train['SEGMENT_ID'] = train['SEGMENT_ID'].astype(str).map(Config.SEGMENT_MAPPING)
prod['SEGMENT_ID'] = prod['SEGMENT_ID'].astype(str).map(Config.SEGMENT_MAPPING)

# Обработка неизвестных сегментов
train['SEGMENT_ID'] = train['SEGMENT_ID'].fillna('large_and_middle')
prod['SEGMENT_ID'] = prod['SEGMENT_ID'].fillna('large_and_middle')

print("\n" + "="*70)
print("НОВЫЕ ОБЪЕДИНЕННЫЕ СЕГМЕНТЫ (2 сегмента)")
print("="*70)
print(train['SEGMENT_ID'].value_counts().sort_index())

print("\nДетальное распределение в TRAIN:")
for segment in sorted(train['SEGMENT_ID'].unique()):
    seg_data = train[train['SEGMENT_ID'] == segment]
    count = len(seg_data)
    clients = seg_data['CLIENT_ID'].nunique()
    pct = 100 * count / len(train)
    avg_margin = seg_data['TARGET_NEXT_MARGIN'].mean()
    print(f"  {segment:20s}: {count:>10,} записей ({pct:>5.1f}%), {clients:>8,} клиентов, avg margin: {avg_margin:>12,.0f}")

In [ ]:
# %% ПРЕДОБРАБОТКА
all_categorical = Config.CATEGORICAL_FEATURES + ['SEGMENT_ID']
train_fixed = fix_categorical_features(train, all_categorical)
prod_fixed = fix_categorical_features(prod, all_categorical)

ALL_FEATURES = ['SEGMENT_ID'] + Config.BASE_FEATURES + Config.CATEGORICAL_FEATURES
available_features = [f for f in ALL_FEATURES if f in train_fixed.columns]

numeric_features = [f for f in available_features if f not in all_categorical]
for col in numeric_features:
    train_fixed[col] = train_fixed[col].fillna(0.0)
    if col in prod_fixed.columns:
        prod_fixed[col] = prod_fixed[col].fillna(0.0)

print(f"Фичи готовы: {len(available_features)}")
print(f"Числовые фичи: {len(numeric_features)}")
print(f"Категориальные фичи: {len(Config.CATEGORICAL_FEATURES)}")
print(f"Сегменты: {sorted(train_fixed['SEGMENT_ID'].unique())}")

In [ ]:
# %% КЛАСС ДЛЯ ОБУЧЕНИЯ С OOT
class SimpleCLTV_OOT:
    """Упрощенная версия с OOT выборкой"""
    
    def __init__(self, catboost_params):
        self.models = {}
        self.segment_stats = {}
        self.feature_importance = {}
        self.catboost_params = catboost_params
        
    def train_segment(self, segment_id, X_train, y_train, X_val, y_val, X_oot, y_oot, cat_features=None):
        """Обучение модели для одного сегмента с OOT"""
        print(f"\nОбучение сегмента: {segment_id}")
        print(f"  Train samples: {len(X_train):,}")
        print(f"  Val samples: {len(X_val):,}")
        print(f"  OOT samples: {len(X_oot):,}")
        
        # Определение индексов категориальных признаков
        cat_indices = []
        if cat_features:
            features_list = X_train.columns.tolist()
            for cat_feat in cat_features:
                if cat_feat in features_list:
                    cat_indices.append(features_list.index(cat_feat))
        
        # Создание Pool
        train_pool = Pool(X_train, y_train, cat_features=cat_indices)
        val_pool = Pool(X_val, y_val, cat_features=cat_indices)
        oot_pool = Pool(X_oot, y_oot, cat_features=cat_indices)
        
        # Обучение на train, валидация на val
        model = CatBoostRegressor(**self.catboost_params)
        model.fit(train_pool, eval_set=val_pool, use_best_model=True)
        
        # Метрики на всех выборках
        train_pred = model.predict(X_train)
        val_pred = model.predict(X_val)
        oot_pred = model.predict(X_oot)
        
        metrics = {
            # Train
            'train_r2': r2_score(y_train, train_pred),
            'train_rmse': np.sqrt(mean_squared_error(y_train, train_pred)),
            'train_mae': mean_absolute_error(y_train, train_pred),
            'train_samples': len(X_train),
            # Validation
            'val_r2': r2_score(y_val, val_pred),
            'val_rmse': np.sqrt(mean_squared_error(y_val, val_pred)),
            'val_mae': mean_absolute_error(y_val, val_pred),
            'val_samples': len(X_val),
            # OOT (Test)
            'oot_r2': r2_score(y_oot, oot_pred),
            'oot_rmse': np.sqrt(mean_squared_error(y_oot, oot_pred)),
            'oot_mae': mean_absolute_error(y_oot, oot_pred),
            'oot_samples': len(X_oot),
        }
        
        # Feature importance
        importance_df = pd.DataFrame({
            'feature': X_train.columns,
            'importance': model.feature_importances_
        }).sort_values('importance', ascending=False)
        
        print(f"\n  Результаты:")
        print(f"    Train R²: {metrics['train_r2']:.4f}, RMSE: {metrics['train_rmse']:.2f}, MAE: {metrics['train_mae']:.2f}")
        print(f"    Val   R²: {metrics['val_r2']:.4f}, RMSE: {metrics['val_rmse']:.2f}, MAE: {metrics['val_mae']:.2f}")
        print(f"    🎯 OOT R²: {metrics['oot_r2']:.4f}, RMSE: {metrics['oot_rmse']:.2f}, MAE: {metrics['oot_mae']:.2f}")
        
        print(f"\n  Топ-10 важных фичей:")
        for idx, row in importance_df.head(10).iterrows():
            print(f"    {row['feature']:30s}: {row['importance']:.2f}")
        
        self.models[segment_id] = model
        self.segment_stats[segment_id] = metrics
        self.feature_importance[segment_id] = importance_df
        
        return model, metrics
    
    def predict(self, segment_id, X):
        """Предсказание для сегмента"""
        if segment_id not in self.models:
            raise ValueError(f"Нет модели для сегмента {segment_id}")
        
        X_pred = X.drop('SEGMENT_ID', axis=1) if 'SEGMENT_ID' in X.columns else X
        return self.models[segment_id].predict(X_pred)

print("Класс SimpleCLTV_OOT создан")

In [ ]:
# %% ПОДГОТОВКА ДАННЫХ С РАЗДЕЛЕНИЕМ НА 3 ВЫБОРКИ
print("\n" + "="*70)
print("ПОДГОТОВКА ДАННЫХ: TRAIN / VALIDATION / OOT")
print("="*70)

# Стабилизация таргета
train_fixed['target_stable'] = stabilize_target(train_fixed['TARGET_NEXT_MARGIN'])

# Разделение на train / val / oot по времени
train_mask = pd.to_datetime(train_fixed['MONTH_END']) <= pd.to_datetime(Config.TRAIN_CUTOFF)
val_mask = (pd.to_datetime(train_fixed['MONTH_END']) > pd.to_datetime(Config.TRAIN_CUTOFF)) & \
           (pd.to_datetime(train_fixed['MONTH_END']) <= pd.to_datetime(Config.VALIDATION_CUTOFF))
oot_mask = pd.to_datetime(train_fixed['MONTH_END']) > pd.to_datetime(Config.VALIDATION_CUTOFF)

print(f"\nОбщее разделение:")
print(f"  Train: {train_mask.sum():,} записей ({100*train_mask.sum()/len(train_fixed):.1f}%)")
print(f"  Val:   {val_mask.sum():,} записей ({100*val_mask.sum()/len(train_fixed):.1f}%)")
print(f"  OOT:   {oot_mask.sum():,} записей ({100*oot_mask.sum()/len(train_fixed):.1f}%)")

# Подготовка данных по сегментам
segment_data = {}

for segment_id in sorted(train_fixed['SEGMENT_ID'].unique()):
    seg_mask = train_fixed['SEGMENT_ID'] == segment_id
    
    seg_train_mask = seg_mask & train_mask
    seg_val_mask = seg_mask & val_mask
    seg_oot_mask = seg_mask & oot_mask
    
    features_for_segment = [f for f in available_features if f != 'SEGMENT_ID']
    
    X_train = train_fixed[seg_train_mask][features_for_segment]
    y_train = train_fixed[seg_train_mask]['target_stable']
    X_val = train_fixed[seg_val_mask][features_for_segment]
    y_val = train_fixed[seg_val_mask]['target_stable']
    X_oot = train_fixed[seg_oot_mask][features_for_segment]
    y_oot = train_fixed[seg_oot_mask]['target_stable']
    
    segment_data[segment_id] = {
        'X_train': X_train,
        'y_train': y_train,
        'X_val': X_val,
        'y_val': y_val,
        'X_oot': X_oot,
        'y_oot': y_oot
    }
    
    print(f"\nСегмент {segment_id}:")
    print(f"  Train: {len(X_train):,} записей")
    print(f"  Val:   {len(X_val):,} записей")
    print(f"  OOT:   {len(X_oot):,} записей")

print("\nДанные подготовлены")

In [ ]:
# %% ОБУЧЕНИЕ МОДЕЛЕЙ С OOT
print("\n" + "="*70)
print("НАЧИНАЕМ ОБУЧЕНИЕ МОДЕЛЕЙ С OOT ВАЛИДАЦИЕЙ")
print("="*70)

cltv_model = SimpleCLTV_OOT(catboost_params=Config.CATBOOST_PARAMS)

for segment_id in sorted(segment_data.keys()):
    print(f"\n{'='*70}")
    print(f"СЕГМЕНТ: {segment_id.upper()}")
    print(f"{'='*70}")
    
    data = segment_data[segment_id]
    
    model, metrics = cltv_model.train_segment(
        segment_id=segment_id,
        X_train=data['X_train'],
        y_train=data['y_train'],
        X_val=data['X_val'],
        y_val=data['y_val'],
        X_oot=data['X_oot'],
        y_oot=data['y_oot'],
        cat_features=Config.CATEGORICAL_FEATURES
    )

print("\n" + "="*70)
print("ОБУЧЕНИЕ ЗАВЕРШЕНО")
print("="*70)

In [ ]:
# %% СВОДКА РЕЗУЛЬТАТОВ С OOT
print("\n" + "="*70)
print("СВОДКА ПО МОДЕЛЯМ (Train / Val / OOT)")
print("="*70)

summary_data = []
for segment_id, stats in cltv_model.segment_stats.items():
    summary_data.append({
        'Сегмент': segment_id,
        'Train_R2': f"{stats['train_r2']:.4f}",
        'Val_R2': f"{stats['val_r2']:.4f}",
        'OOT_R2': f"{stats['oot_r2']:.4f}",
        'Train_RMSE': f"{stats['train_rmse']:.2f}",
        'Val_RMSE': f"{stats['val_rmse']:.2f}",
        'OOT_RMSE': f"{stats['oot_rmse']:.2f}",
        'Train_MAE': f"{stats['train_mae']:.2f}",
        'Val_MAE': f"{stats['val_mae']:.2f}",
        'OOT_MAE': f"{stats['oot_mae']:.2f}",
        'Train_N': f"{stats['train_samples']:,}",
        'Val_N': f"{stats['val_samples']:,}",
        'OOT_N': f"{stats['oot_samples']:,}"
    })

summary_df = pd.DataFrame(summary_data)
print("\n" + summary_df.to_string(index=False))

# Средние метрики
avg_train_r2 = np.mean([stats['train_r2'] for stats in cltv_model.segment_stats.values()])
avg_val_r2 = np.mean([stats['val_r2'] for stats in cltv_model.segment_stats.values()])
avg_oot_r2 = np.mean([stats['oot_r2'] for stats in cltv_model.segment_stats.values()])
avg_oot_rmse = np.mean([stats['oot_rmse'] for stats in cltv_model.segment_stats.values()])
avg_oot_mae = np.mean([stats['oot_mae'] for stats in cltv_model.segment_stats.values()])

print(f"\n{'='*70}")
print("СРЕДНИЕ МЕТРИКИ:")
print(f"  Train R²: {avg_train_r2:.4f}")
print(f"  Val R²:   {avg_val_r2:.4f}")
print(f"  🎯 OOT R²:   {avg_oot_r2:.4f}  ← ФИНАЛЬНАЯ ОЦЕНКА")
print(f"  OOT RMSE: {avg_oot_rmse:.2f}")
print(f"  OOT MAE:  {avg_oot_mae:.2f}")
print(f"{'='*70}")

# Анализ стабильности
print(f"\nАНАЛИЗ СТАБИЛЬНОСТИ:")
for segment_id, stats in cltv_model.segment_stats.items():
    train_val_diff = abs(stats['train_r2'] - stats['val_r2'])
    val_oot_diff = abs(stats['val_r2'] - stats['oot_r2'])
    print(f"  {segment_id}:")
    print(f"    |Train R² - Val R²|: {train_val_diff:.4f}")
    print(f"    |Val R² - OOT R²|:  {val_oot_diff:.4f}")
    if val_oot_diff < 0.05:
        print(f"    ✅ Стабильная модель (разница < 0.05)")
    elif val_oot_diff < 0.10:
        print(f"    ⚠️  Умеренная деградация (разница < 0.10)")
    else:
        print(f"    ❌ Сильная деградация (разница >= 0.10)")

In [ ]:
# %% АНАЛИЗ OOT ВЫБОРКИ (ДЕТАЛЬНО)
print("\n" + "="*70)
print("ДЕТАЛЬНЫЙ АНАЛИЗ OOT ВЫБОРКИ (ФИНАЛЬНЫЙ ТЕСТ)")
print("="*70)

for segment_id in sorted(cltv_model.models.keys()):
    print(f"\nСегмент: {segment_id}")
    
    data = segment_data[segment_id]
    y_oot_pred_stable = cltv_model.predict(segment_id, data['X_oot'])
    
    # Обратная трансформация
    y_oot_true = inverse_stabilize_target(data['y_oot'])
    y_oot_pred = inverse_stabilize_target(y_oot_pred_stable)
    
    # Статистика
    print(f"\n  Истинные значения (OOT):")
    print(f"    Mean:   {y_oot_true.mean():,.0f}")
    print(f"    Median: {y_oot_true.median():,.0f}")
    print(f"    Std:    {y_oot_true.std():,.0f}")
    
    print(f"\n  Предсказанные значения (OOT):")
    print(f"    Mean:   {y_oot_pred.mean():,.0f}")
    print(f"    Median: {np.median(y_oot_pred):,.0f}")
    print(f"    Std:    {y_oot_pred.std():,.0f}")
    
    # Метрики на оригинальной шкале
    r2_orig = r2_score(y_oot_true, y_oot_pred)
    rmse_orig = np.sqrt(mean_squared_error(y_oot_true, y_oot_pred))
    mae_orig = mean_absolute_error(y_oot_true, y_oot_pred)
    mape = np.mean(np.abs((y_oot_true - y_oot_pred) / (y_oot_true + 1e-6))) * 100
    
    print(f"\n  🎯 ФИНАЛЬНЫЕ МЕТРИКИ (оригинальная шкала):")
    print(f"    R²:   {r2_orig:.4f}")
    print(f"    RMSE: {rmse_orig:,.0f} руб")
    print(f"    MAE:  {mae_orig:,.0f} руб")
    print(f"    MAPE: {mape:.1f}%")

In [ ]:
# %% СОХРАНЕНИЕ МОДЕЛЕЙ
print("\n" + "="*70)
print("СОХРАНЕНИЕ МОДЕЛЕЙ")
print("="*70)

save_path = Config.MODEL_DIR / Config.MODEL_VERSION

# Сохранение CatBoost моделей
for segment_id, model in cltv_model.models.items():
    model_file = save_path / f"model_{segment_id}.cbm"
    model.save_model(str(model_file))
    print(f"  ✓ Сохранена модель: {model_file}")

# Сохранение метрик
summary_df.to_csv(save_path / "metrics_summary_oot.csv", index=False)
print(f"  ✓ Сохранены метрики: {save_path / 'metrics_summary_oot.csv'}")

# Сохранение feature importance
for segment_id, importance_df in cltv_model.feature_importance.items():
    importance_file = save_path / f"feature_importance_{segment_id}.csv"
    importance_df.to_csv(importance_file, index=False)
    print(f"  ✓ Сохранена важность фичей: {importance_file}")

# Метаданные
metadata = {
    'version': Config.MODEL_VERSION,
    'train_date': datetime.now().isoformat(),
    'segments': list(cltv_model.models.keys()),
    'features': available_features,
    'segment_mapping': Config.SEGMENT_MAPPING,
    'catboost_params': Config.CATBOOST_PARAMS,
    'data_split': {
        'train_cutoff': Config.TRAIN_CUTOFF,
        'validation_cutoff': Config.VALIDATION_CUTOFF,
        'train_pct': float(train_mask.sum() / len(train_fixed)),
        'val_pct': float(val_mask.sum() / len(train_fixed)),
        'oot_pct': float(oot_mask.sum() / len(train_fixed))
    },
    'metrics': {seg: {k: float(v) if isinstance(v, (int, float, np.number)) else v 
                     for k, v in stats.items()} 
               for seg, stats in cltv_model.segment_stats.items()}
}

with open(save_path / "metadata.json", 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
print(f"  ✓ Сохранены метаданные: {save_path / 'metadata.json'}")

# Сохранение объекта модели
with open(save_path / "model_object.pkl", 'wb') as f:
    pickle.dump(cltv_model, f)
print(f"  ✓ Сохранен объект модели: {save_path / 'model_object.pkl'}")

print(f"\n{'='*70}")
print(f"✅ ВСЕ МОДЕЛИ СОХРАНЕНЫ В: {save_path}")
print(f"{'='*70}")

In [ ]:
# %% ВИЗУАЛИЗАЦИЯ МЕТРИК
print("\n" + "="*70)
print("ВИЗУАЛИЗАЦИЯ R² НА ВСЕХ ВЫБОРКАХ")
print("="*70)

# График сравнения R² на Train/Val/OOT
fig, ax = plt.subplots(figsize=(12, 6))

segments = list(cltv_model.segment_stats.keys())
x = np.arange(len(segments))
width = 0.25

train_r2 = [cltv_model.segment_stats[seg]['train_r2'] for seg in segments]
val_r2 = [cltv_model.segment_stats[seg]['val_r2'] for seg in segments]
oot_r2 = [cltv_model.segment_stats[seg]['oot_r2'] for seg in segments]

ax.bar(x - width, train_r2, width, label='Train R²', alpha=0.8)
ax.bar(x, val_r2, width, label='Val R²', alpha=0.8)
ax.bar(x + width, oot_r2, width, label='OOT R² (Test)', alpha=0.8, color='green')

ax.set_xlabel('Сегмент')
ax.set_ylabel('R²')
ax.set_title('Сравнение R² на Train / Validation / OOT выборках')
ax.set_xticks(x)
ax.set_xticklabels(segments)
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.axhline(y=0.8, color='r', linestyle='--', alpha=0.5, label='R²=0.8')

# Добавление значений на столбцы
for i, (t, v, o) in enumerate(zip(train_r2, val_r2, oot_r2)):
    ax.text(i - width, t + 0.02, f'{t:.3f}', ha='center', va='bottom', fontsize=9)
    ax.text(i, v + 0.02, f'{v:.3f}', ha='center', va='bottom', fontsize=9)
    ax.text(i + width, o + 0.02, f'{o:.3f}', ha='center', va='bottom', fontsize=9, weight='bold')

plt.tight_layout()
plt.savefig(save_path / 'r2_comparison.png', dpi=100, bbox_inches='tight')
print(f"\n✅ График сохранен: {save_path / 'r2_comparison.png'}")
plt.show()

In [ ]:
# %% ИТОГИ С OOT
print("\n" + "="*70)
print("ИТОГОВАЯ ИНФОРМАЦИЯ")
print("="*70)

print(f"\n📊 РЕЗУЛЬТАТЫ ОБУЧЕНИЯ С OOT:")
print(f"  ✓ Обучено моделей: {len(cltv_model.models)}")
print(f"  ✓ Сегменты: {', '.join(sorted(cltv_model.models.keys()))}")
print(f"\n  📈 Средние метрики:")
print(f"    Train R²: {avg_train_r2:.4f}")
print(f"    Val R²:   {avg_val_r2:.4f}")
print(f"    🎯 OOT R²:   {avg_oot_r2:.4f} ← ФИНАЛЬНАЯ ОЦЕНКА КАЧЕСТВА")

print(f"\n💾 СОХРАНЕННЫЕ ФАЙЛЫ:")
print(f"  ✓ Директория: {save_path}")
print(f"  ✓ Модели CatBoost: {len(cltv_model.models)} файлов")
print(f"  ✓ Метрики: metrics_summary_oot.csv")
print(f"  ✓ Метаданные: metadata.json")
print(f"  ✓ Графики: r2_comparison.png")

print(f"\n📋 РАЗДЕЛЕНИЕ ДАННЫХ:")
print(f"  ✓ Train: {train_mask.sum():,} ({100*train_mask.sum()/len(train_fixed):.1f}%)")
print(f"  ✓ Val:   {val_mask.sum():,} ({100*val_mask.sum()/len(train_fixed):.1f}%)")
print(f"  ✓ OOT:   {oot_mask.sum():,} ({100*oot_mask.sum()/len(train_fixed):.1f}%)")

print(f"\n{'='*70}")
print("✅ ОБУЧЕНИЕ С OOT ВАЛИДАЦИЕЙ ЗАВЕРШЕНО УСПЕШНО!")
print(f"{'='*70}")
print(f"\n💡 OOT R² = {avg_oot_r2:.4f} - это самая честная оценка вашей модели!")
print(f"   Используйте эту метрику для принятия решения о запуске в продакшн.")